## Experiment 3: Thinking vs Non-Thinking Mode

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# ── INSTALL DEPENDENCIES ─────────────────────────────────────────
!pip install -q transformers accelerate datasets sacrebleu evaluate \
    sentencepiece protobuf bitsandbytes peft trl

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — switch runtime!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# ── GLOBAL CONFIG ────────────────────────────────────────────────

MODEL_NAME = 'Qwen/Qwen3-0.6B'

LANG_PAIRS = [
    ('eng_Latn', 'fra_Latn', 'English', 'French'),   # High resource
    ('fra_Latn', 'eng_Latn', 'French', 'English'),   # Reverse
    ('eng_Latn', 'xho_Latn', 'English', 'Xhosa'),    # Low resource
]

N_EVAL = 50   # Number of test sentences.

import json, re, random, pandas as pd, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm.notebook import tqdm

random.seed(42)
np.random.seed(42)

In [ ]:
# ── CELL 3: LOAD MODEL ───────────────────────────────────────────────────
import torch

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()
print('Model loaded!')

In [ ]:
# ── UTILITY FUNCTIONS (batched / parallelized generation) ─────────────

def load_flores(src_lang, tgt_lang, split='devtest', n=50):
    src_ds = load_dataset('openlanguagedata/flores_plus', src_lang, split=split)
    tgt_ds = load_dataset('openlanguagedata/flores_plus', tgt_lang, split=split)
    return [ex['text'] for ex in src_ds][:n], [ex['text'] for ex in tgt_ds][:n]

# Quick test
test_src, test_ref = load_flores('eng_Latn', 'fra_Latn', split='devtest', n=3)
for s, r in zip(test_src, test_ref):
    print(f"EN: {s}\nFR: {r}\n")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # important for batched generation with decoder-only LMs

def make_translation_prompt(
    src_text: str,
    src_name: str,
    tgt_name: str,
    model_family: str = "generic",
    thinking: bool = True,
) -> str:
    prompt = (
        f"Please write a high-quality {tgt_name} translation of the following {src_name} sentence\n"
        f"\"{src_text}\"\n"
        "Please provide only the translation, nothing more.\n"
    )

    # Paper's Qwen non-thinking mode
    if model_family.lower().startswith("qwen") and not thinking:
        prompt += "<think>\n\n</think>\n"

    return prompt


import re

def _extract_translation(decoded_text: str) -> str:
    if not decoded_text:
        return ""

    text = decoded_text.strip()

    # Remove thinking traces
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

    # Prefer content after "Final Translation" if present
    m = re.search(r"Final Translation\s*:?\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    # Drop any remaining tags
    text = re.sub(r"</?[^>]+>", "", text).strip()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text.strip(" \"'")


def _build_chat_prompts(sources, prompt_fn, enable_thinking=False):
    prompts = []
    for src in sources:
        msgs = [{'role': 'user', 'content': prompt_fn(src)}]
        prompts.append(
            tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
        )
    return prompts


def translate(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=256,
    enable_thinking=False,
    batch_size=1024,
):
    """
    Batched generation for much better GPU utilization on Colab.
    Falls back to a smaller batch automatically if CUDA OOM happens.
    """
    prompts = _build_chat_prompts(sources, prompt_fn, enable_thinking=enable_thinking)
    translations = []
    i = 0

    pbar = tqdm(total=len(prompts))
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)

        while current_bs >= 1:
            batch_prompts = prompts[i:i + current_bs]
            try:
                inputs = tokenizer(
                    batch_prompts,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=None,
                        top_p=None,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                # IMPORTANT: generated tokens start after the padded input width,
                # not after each sample's non-pad length.
                prompt_width = inputs['input_ids'].shape[1]

                for row in outputs:
                    generated_tokens = row[prompt_width:]
                    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                    translations.append(_extract_translation(decoded))

                i += current_bs
                pbar.update(current_bs)
                break

            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                current_bs //= 2
                if current_bs == 0:
                    raise RuntimeError(
                        'Out of memory even with batch_size=1. '
                        'Lower max_new_tokens or use a smaller model.'
                    )

    pbar.close()
    return translations


def score(hyps, refs):
    bleu = evaluate.load('sacrebleu')
    chrf = evaluate.load('chrf')
    b = bleu.compute(predictions=hyps, references=[[r] for r in refs])['score']
    c = chrf.compute(predictions=hyps, references=[[r] for r in refs], word_order=2)['score']
    return round(b, 2), round(c, 2)

print('Utilities loaded! Batched translation is enabled.')

In [ ]:
# Quick verification
test_src, test_ref = load_flores('eng_Latn', 'fra_Latn', split='devtest', n=11)
for s, r in zip(test_src, test_ref):
    print(f"EN: {s}\nFR: {r}\n\n")

In [ ]:
# ── DEBUG FUNCTIONS (batched / parallelized generation) ─────────────

import re
from math import ceil

def contains_think_block(text: str) -> bool:
    if not text:
        return False
    return re.search(r"<think>.*?</think>", text, flags=re.DOTALL | re.IGNORECASE) is not None


def extract_think_block(text: str) -> str | None:
    if not text:
        return None
    m = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return None


def translate_with_raw_outputs(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=512,
    enable_thinking=True,
    batch_size=32,
):
    """
    Returns:
        raw_outputs: decoded model generations before translation extraction
        hyps: extracted translations after _extract_translation(...)
    """
    prompts = _build_chat_prompts(
        sources,
        prompt_fn,
        enable_thinking=enable_thinking,
    )

    raw_outputs = []
    hyps = []
    i = 0

    pbar = tqdm(total=len(prompts))
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)

        while current_bs >= 1:
            batch_prompts = prompts[i:i + current_bs]

            try:
                inputs = tokenizer(
                    batch_prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=None,
                        top_p=None,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                prompt_width = inputs["input_ids"].shape[1]

                for row in outputs:
                    generated_tokens = row[prompt_width:]
                    raw_text = tokenizer.decode(
                        generated_tokens,
                        skip_special_tokens=True
                    ).strip()
                    raw_outputs.append(raw_text)
                    hyps.append(_extract_translation(raw_text))

                i += current_bs
                pbar.update(current_bs)
                break

            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                current_bs //= 2
                if current_bs < 1:
                    raise

    pbar.close()
    return raw_outputs, hyps


def print_raw_output_examples(
    sources,
    references,
    raw_outputs,
    hyps,
    mode_label,
    n_examples=3,
    max_chars=2500,
):
    print(f"\n{'='*100}")
    print(f"RAW OUTPUT EXAMPLES — {mode_label}")
    print(f"{'='*100}")

    for i, (src, ref, raw, hyp) in enumerate(zip(sources[:n_examples], references[:n_examples], raw_outputs[:n_examples], hyps[:n_examples])):
        has_think = contains_think_block(raw)
        think_text = extract_think_block(raw)

        print(f"\n--- Example {i+1} [{mode_label}] ---")
        print(f"SOURCE:\n{src}\n")
        print(f"REFERENCE:\n{ref}\n")
        print(f"HAS <think> BLOCK: {has_think}\n")

        raw_to_show = raw[:max_chars]
        if len(raw) > max_chars:
            raw_to_show += "\n...[truncated]..."

        print(f"RAW MODEL OUTPUT:\n{raw_to_show}\n")

        if think_text:
            think_to_show = think_text[:max_chars]
            if len(think_text) > max_chars:
                think_to_show += "\n...[truncated]..."
            print(f"EXTRACTED THINKING:\n{think_to_show}\n")

        print(f"EXTRACTED TRANSLATION:\n{hyp}\n")

In [ ]:
# ── EXP 3: THINKING vs NON-THINKING ──────────────────

exp3_results = []
exp3_examples = []

N_EXAMPLES_TO_STORE = 20
N_EXAMPLES_TO_PRINT = 3
N_RAW_DEBUG_PRINT = 2

for src_lang, tgt_lang, src_name, tgt_name in LANG_PAIRS:
    print(f'\n{"="*100}')
    print(f'{src_name} → {tgt_name}')
    print(f'{"="*100}')

    sources, references = load_flores(src_lang, tgt_lang, n=N_EVAL)

    for thinking, label in [(True, 'Thinking'), (False, 'Non-Thinking')]:
        fn = lambda s, sn=src_name, tn=tgt_name, t=thinking: make_translation_prompt(
            s, sn, tn, "qwen", t
        )

        max_new_tokens = 3500
        if 'Xhosa' in src_name or 'Xhosa' in tgt_name:
          max_new_tokens = 1024

        raw_outputs, hyps = translate_with_raw_outputs(
            model,
            tokenizer,
            sources,
            fn,
            max_new_tokens=max_new_tokens,
            enable_thinking=thinking,
            batch_size=1024,
        )

        b, c = score(hyps, references)

        think_flags = [contains_think_block(x) for x in raw_outputs]
        think_count = sum(think_flags)
        think_ratio = think_count / len(raw_outputs) if raw_outputs else 0.0

        print(
            f'\n  {label:15s} → BLEU: {b:.2f} | chrF++: {c:.2f} '
            f'| with <think>: {think_count}/{len(raw_outputs)} ({100*think_ratio:.1f}%)'
        )

        # Print a few raw outputs so we can inspect reasoning / extra text
        print_raw_output_examples(
            sources=sources,
            references=references,
            raw_outputs=raw_outputs,
            hyps=hyps,
            mode_label=f"{src_name} → {tgt_name} | {label}",
            n_examples=N_RAW_DEBUG_PRINT,
        )

        examples = []
        for i, (src, ref, raw, hyp, has_think) in enumerate(
            zip(
                sources[:N_EXAMPLES_TO_STORE],
                references[:N_EXAMPLES_TO_STORE],
                raw_outputs[:N_EXAMPLES_TO_STORE],
                hyps[:N_EXAMPLES_TO_STORE],
                think_flags[:N_EXAMPLES_TO_STORE],
            )
        ):
            ex = {
                'example_id': i,
                'source': src,
                'reference': ref,
                'raw_output': raw,
                'contains_think_block': has_think,
                'think_block': extract_think_block(raw),
                'hypothesis': hyp,
            }
            examples.append(ex)

            exp3_examples.append({
                'src': src_name,
                'tgt': tgt_name,
                'mode': label,
                'example_id': i,
                'source': src,
                'reference': ref,
                'raw_output': raw,
                'contains_think_block': has_think,
                'think_block': extract_think_block(raw),
                'hypothesis': hyp,
            })

        exp3_results.append({
            'src': src_name,
            'tgt': tgt_name,
            'mode': label,
            'bleu': b,
            'chrf': c,
            'n_outputs': len(raw_outputs),
            'n_with_think_block': think_count,
            'prop_with_think_block': think_ratio,
            'examples': examples,
        })

# Summary table
df3 = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'examples'}
    for r in exp3_results
])

df3_examples = pd.DataFrame(exp3_examples)

print('\n' + '='*100)
print('SUMMARY RESULTS')
print('='*100)
print(df3.to_string(index=False))

# Optional aggregated summary across directions
df3_think_summary = (
    df3.groupby('mode', as_index=False)
      .agg({
          'n_outputs': 'sum',
          'n_with_think_block': 'sum',
          'bleu': 'mean',
          'chrf': 'mean',
      })
)

df3_think_summary['prop_with_think_block'] = (
    df3_think_summary['n_with_think_block'] / df3_think_summary['n_outputs']
)

print('\n' + '='*100)
print('THINK BLOCK SUMMARY BY MODE')
print('='*100)
print(df3_think_summary.to_string(index=False))

# Save everything
with open('results_exp3.json', 'w', encoding='utf-8') as f:
    json.dump(exp3_results, f, indent=2, ensure_ascii=False)

df3.to_csv('results_exp3_summary.csv', index=False, encoding='utf-8')
df3_examples.to_csv('results_exp3_examples.csv', index=False, encoding='utf-8')
df3_think_summary.to_csv('results_exp3_think_block_summary.csv', index=False, encoding='utf-8')